# Linear Regression in Julia

A small, self-contained walkthrough of ordinary least squares (OLS) using only the Julia standard library.

We fit the model

$$y = X\beta + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2 I),$$

and recover the coefficient estimate that minimises the residual sum of squares,

$$\hat{\beta} = \arg\min_{\beta}\; \lVert y - X\beta \rVert_2^2.$$

Only `Random`, `Statistics`, and `LinearAlgebra` (all stdlib) are required.

In [ ]:
using Random
using Statistics
using LinearAlgebra

## 1. Generate synthetic data

We build a design matrix `X` with an intercept column of ones plus two predictors, then generate responses from known true coefficients with Gaussian noise.

In [ ]:
rng = MersenneTwister(20260531)

n = 200                      # number of observations
x1 = randn(rng, n)
x2 = randn(rng, n)

# Design matrix with an intercept column.
X = hcat(ones(n), x1, x2)

# True coefficients: intercept = 2.0, slopes = 1.5 and -0.8.
β_true = [2.0, 1.5, -0.8]
σ = 0.5

y = X * β_true .+ σ .* randn(rng, n)

size(X), size(y)

## 2. Fit by ordinary least squares

The numerically sound way to solve the normal equations in Julia is the backslash operator `\`, which uses a QR factorisation rather than forming `inv(X'X)` explicitly:

$$\hat{\beta} = (X^\top X)^{-1} X^\top y \quad\Longleftrightarrow\quad \beta = X \backslash y.$$

In [ ]:
β_hat = X \ y

println("Estimated coefficients:")
for (name, b) in zip(["intercept", "x1", "x2"], β_hat)
    println("  ", rpad(name, 10), round(b; digits = 4))
end

## 3. Residuals, variance, and standard errors

With $p$ parameters and $n$ observations, the unbiased noise-variance estimate is

$$\hat{\sigma}^2 = \frac{\lVert y - X\hat{\beta}\rVert_2^2}{n - p},$$

and the coefficient covariance is $\hat{\sigma}^2 (X^\top X)^{-1}$. We obtain $(X^\top X)^{-1}$ from a Cholesky factor instead of an explicit inverse.

In [ ]:
ŷ = X * β_hat
resid = y .- ŷ

n, p = size(X)
σ²_hat = sum(abs2, resid) / (n - p)

# Coefficient covariance via the Cholesky factor of X'X (no explicit inverse).
C = cholesky(Symmetric(X' * X))
covβ = σ²_hat .* inv(C)          # inv(::Cholesky) is a stable triangular solve
se = sqrt.(diag(covβ))

println("σ̂  = ", round(sqrt(σ²_hat); digits = 4), "  (true σ = ", σ, ")")
println("\nCoefficient   estimate     std.err     t-stat")
for (name, b, s) in zip(["intercept", "x1", "x2"], β_hat, se)
    println("  ", rpad(name, 12), rpad(round(b; digits = 4), 12),
            rpad(round(s; digits = 4), 12), round(b / s; digits = 3))
end

## 4. Goodness of fit ($R^2$)

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}.$$

In [ ]:
ss_res = sum(abs2, resid)
ss_tot = sum(abs2, y .- mean(y))
r2 = 1 - ss_res / ss_tot
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p)

println("R²          = ", round(r2; digits = 4))
println("adjusted R² = ", round(r2_adj; digits = 4))

## 5. Optional: visualise the fit

If `Plots.jl` is available in your environment, the cell below shows predicted vs. observed values. It is wrapped in a `try`/`catch` so the notebook still runs end-to-end without the dependency.

Install it with `import Pkg; Pkg.add("Plots")` if you want the figure.

In [ ]:
try
    @eval using Plots
    scatter(ŷ, y; xlabel = "predicted ŷ", ylabel = "observed y",
            label = "data", legend = :topleft, title = "OLS fit")
    lo, hi = extrema(vcat(ŷ, y))
    plot!([lo, hi], [lo, hi]; label = "y = ŷ", lw = 2, color = :red)
catch err
    @info "Plots.jl not available — skipping figure" exception = err
end

## Summary

- `X \ y` is the idiomatic, numerically stable OLS solver in Julia (QR-based; avoids forming `inv(X'X)`).
- Variance and standard errors follow from the residuals and the Cholesky factor of $X^\top X$.
- The recovered coefficients should sit close to the true `[2.0, 1.5, -0.8]`, within the standard errors.

For real modelling work with formulas, categorical predictors, and inference tables, reach for [`GLM.jl`](https://github.com/JuliaStats/GLM.jl); for mixed-effects models, [`MixedModels.jl`](https://github.com/JuliaStats/MixedModels.jl).